## Allocation test version 1
This test doesn't import the module with the functions to allocate crews and create controls, instead has the block
of code of those functions written within this notebook.

In [1]:
#import sys
#!{sys.executable} -m pip install import_ipynb

In [2]:
#import import_ipynb
#from BPDRR_crew_allocation import allocate_crews
from pprint import pprint
import random
import pandas as pd
import numpy as np

In [3]:
def allocate_crews(reparations, dmatrix, indexes, n_teams):
    missing_repairs = set(indexes) - set(reparations)
    if missing_repairs:
        raise KeyError(f"IDs missing from reparations: {sorted(missing_repairs)}")

    missing_matrix = set(indexes) - set(dmatrix.index)
    if missing_matrix:
        raise KeyError(f"IDs missing from travel-time matrix: {sorted(missing_matrix)}")

    # existing allocation logic...
    """
    allocate_crews(reparations, dmatrix, indexes, n_teams):
    
    Description
       This is essentially a greedy load-balancing problem: 
       process jobs in the order given by indexes, and always assign the next job 
       to the crew with the smallest accumulated repair time.
    
    Input Parameters
    -----------------
    reparations : dict
        Dictionary {id: repair_time}
    dmatrix : dict
        Dictionary {id_i: {id_j: travel_time}}
    indexes : list
        List of ids indicating the allocation order, must have the same length of the keys of 'reparations'
    n_teams : int
        Number of crews, how many crews are you sending in the field to repair pipes

    Output / Returns
    ----------------
    dictionary with the pipe ids and the times of interventions for each crew. 
    dict { 'crew_1': { 'pipe_ids': [...], 'time_total': ...},
           'crew_2': { 'pipe_ids': [...], 'time_total': ...}, 
           ...
         }

    Developed by : Mario Castro-Gama, ir. MSc. PhD
    Last update  : 2026-05-20
                   2026-06-15, added travel time constant
                   2026-07-21, added travel tiem as function of 'dmatrix' distance matrix
    
    """

    # if each crew key is a string
    # crews = { f"crew_{i+1}": {'pipe_ids':  [], 
    #                           'time_k':    [], 
    #                           'time_t':    [], 
    #                           'time_0':    [],
    #                           'time_1':    [],
    #                           'time_total': 0,
    #                          } for i in range(n_teams)}

    # if each crew key is an int (starting at 1)
    crews = { i+1: {'pipe_ids':  [], 
                    'time_repair':    [], 
                    'time_travel':   [],
                    'time_0':    [],
                    'time_1':    [],
                    'time_total': 0,
                   } for i in range(n_teams)}

    nrep = len(reparations)

    print('')
    print('organize distances for each crew')
    new_controls = []
    for idx in indexes:
        if idx not in reparations:
            raise KeyError(f"Pipe ID '{idx}' not found in reparations")

        # Find the crew with the minimum accumulated time to allocate the next reparation
        # first available gets chosen
        crew_curr = min(crews, key = lambda c: crews[c]['time_total'])

        # current reparation time
        repair_time = reparations[idx]['t_r']

        # find the travel time between this pipe and the previous one
        if crews[crew_curr]['pipe_ids']==[]:
            travel_time = 0.5  # no previous pipe so give it 30 minutes
        else:
            # This is estimated from distance matrix 'dmatrix', that matrix is square and static for each damage scenario
            pipe_prev = crews[crew_curr]['pipe_ids'][-1]
            travel_time = dmatrix[pipe_prev][idx]
            print('Crew '+str(crew_curr)+', from '+pipe_prev+'-to-'+idx+' : '+str(travel_time))
        
        # Assign the reparation to each crew
        crews[crew_curr]['pipe_ids'].append(idx)
        crews[crew_curr]['time_repair'].append(repair_time)
        crews[crew_curr]['time_travel'].append(travel_time)
        crews[crew_curr]['time_0'].append(travel_time + crews[crew_curr]['time_total'])
        crews[crew_curr]['time_1'].append(repair_time + crews[crew_curr]['time_0'][-1])
        crews[crew_curr]['time_total'] += repair_time + travel_time

        new_controls.append(f"; Crew {crew_curr} - Pipe {idx}\n")
        new_controls.append(f"LINK {idx}_A CLOSED AT TIME {crews[crew_curr]['time_0'][-1]}\n") # close Pipe_id_<A> when arriving to the location
        new_controls.append(f"LINK {idx}_B CLOSED AT TIME {crews[crew_curr]['time_0'][-1]}\n") # close Pipe_id_<B> when arriving to the location
        new_controls.append(f"LINK {idx} OPEN AT TIME {crews[crew_curr]['time_1'][-1]}\n")     # Open Pipe_id only after time of reparation
        
    return crews, new_controls

In [ ]:
# Damage scenario selection
ds_sel = 'DS5'

df = pd.read_excel(
    "DS_with_full_description.xlsx",
    sheet_name= ds_sel
)

# Repair time discretization (hours)
repair_time_interval = 0.25  # 15 minutes

# Build the reparations dictionary
time_reparation = {}

for _, row in df.iterrows():

    repair_time = float(row["fix time (hours)"])

    # Round up to the nearest interval
    repair_time = np.ceil(repair_time / repair_time_interval) * repair_time_interval

    time_reparation[str(row["Pipe ID"])] = {
        "t_r": repair_time
    }

print(f"{len(time_reparation)} repairs loaded.")
#print(list(time_reparation.items())[5:10])
time_reparation

106 repairs loaded.
[('6005', {'t_r': 5.724455894848762}), ('1252', {'t_r': 5.724455894848762}), ('2408', {'t_r': 5.724455894848762}), ('3094', {'t_r': 5.724455894848762}), ('3293', {'t_r': 5.724455894848762})]


## Example 1
This example uses a dmatrix that uses ramdom values

In [5]:
#n_teams = 3

# Pipe IDs from the reparations dictionary
#pipe_ids = list(time_reparation.keys())

# Number of repairs
#n_rep = len(pipe_ids)

# Random travel times between 0 and 2 hours
#random_matrix = np.random.uniform(0, 2, (n_rep, n_rep))

# Travel from a pipe to itself is zero
#np.fill_diagonal(random_matrix, 0)

# Create DataFrame with pipe IDs
#dmatrix_df = pd.DataFrame(
#    random_matrix,
#    index=pipe_ids,
#    columns=pipe_ids
#)

#dmatrix_df.head()

In [6]:
# generate a random permutation, one would expect to get this directly from the optimization (PYMOO)
#indexes = random.sample(pipe_ids, n_rep)
#print('Show permutation of reparations')
#print(indexes)
#print(len(indexes))

In [7]:
# apply the greedy allocation to the dataset
#crews, new_controls = allocate_crews(time_reparation, dmatrix_df, indexes, n_teams = n_teams)
#print('')
#print('Show the allocation of reparations to crews')
#pprint(crews)
#print('')
#print('[CONTROLS]')
#pprint(new_controls)

In [8]:
#pd.DataFrame({"crews": new_controls}).to_excel(
#    "new_controls.xlsx",
#    index=False
#)

## Example 2
This example uses a dmatrix imported from the files generated by "BPDRR_travel_time_matrix_gen.ipynb"

In [ ]:
# Number of crews
n_teams = 3

# Pipe IDs from the reparations dictionary
pipe_ids = list(time_reparation.keys())

# Number of repairs
n_rep = len(pipe_ids)

In [ ]:
# Import the travel time matrix from the Excel file
dmatrix_df = pd.read_parquet(
    "TravelTime_"+ds_sel+".parquet",
#    sheet_name= ds_sel,
#    index_col=0
)

# Normalize IDs so the repair dictionary and matrix use the same keys
dmatrix_df.index = dmatrix_df.index.map(str)
dmatrix_df.columns = dmatrix_df.columns.map(str)

dmatrix_df.head()

,1951,3414,4988,5251,3404,6005,1252,2408,3094,3293,...,538,5488,5567,5677,5778,5959,6041,854,869,892
1951,0.001335,0.174633,0.059203,0.502553,0.230939,0.574494,0.285973,0.097994,0.227610,0.195529,...,0.248812,0.535616,0.602551,0.510286,0.603958,0.632219,0.577186,0.272680,0.270902,0.270805
3414,0.174633,0.002450,0.159756,0.454119,0.057605,0.526059,0.182835,0.120571,0.053192,0.021110,...,0.135167,0.487181,0.554117,0.461851,0.555523,0.583784,0.528751,0.179511,0.177734,0.157159
4988,0.059203,0.159756,0.006098,0.475991,0.216062,0.547932,0.247441,0.051790,0.212733,0.180652,...,0.216847,0.509054,0.575990,0.483724,0.577396,0.605657,0.550624,0.234147,0.232370,0.238483
5251,0.502553,0.454119,0.475991,0.008997,0.510425,0.091917,0.430311,0.448491,0.507096,0.475014,...,0.399718,0.053039,0.119975,0.027709,0.121381,0.149642,0.094609,0.417018,0.415240,0.421353
3404,0.230939,0.057605,0.216062,0.510425,0.006540,0.582365,0.237347,0.176877,0.069226,0.040278,...,0.189679,0.543487,0.610423,0.518157,0.611829,0.640090,0.585058,0.234023,0.232246,0.211671


In [11]:
indexes = random.sample(pipe_ids, n_rep)
print('Show permutation of reparations')
print(indexes)
print(len(indexes))

Show permutation of reparations
['2910', '4132', '4915', '2430', '2053', '2114', '2409', '6041', '2821', '5105', '3074', '3120', '1807', '4246', '5691', '1477', '3104', '5251', '1568', '2440', '3679', '4584', '2543', '2094', '2408', '1989', '3875', '3414', '3726', '4988', '1464', '5042', '4538', '2881', '1865', '190', '404', '3216', '5778', '2113', '4788', '3125', '3859', '163', '6023', '4177', '1951', '4083', '1186', '4942', '3122', '5959', '2983', '4062', '1402', '6005', '3094', '2307', '1657', '501', '4880', '5567', '4622', '538', '4409', '4038', '1252', '5775', '3041', '3404', '4882', '2216', '4721', '2508', '288', '2535', '762', '3184', '5677', '2730', '4764', '1724', '869', '3449', '3959', '3564', '5871', '2726', '4101', '1373', '3760', '4022', '5488', '2141', '2816', '854', '4594', '2016', '3293', '4507', '1105', '5140', '2001', '4035', '892', '2622']
106


In [12]:
# apply the greedy allocation to the dataset
crews, new_controls = allocate_crews(time_reparation, dmatrix_df, indexes, n_teams = n_teams)
#print('')
#print('Show the allocation of reparations to crews')
#pprint(crews)
print('')
print('[CONTROLS]')
pprint(new_controls)


organize distances for each crew
Crew 4, from 2430-to-2409 : 0.004370731707317074
Crew 6, from 2114-to-6041 : 0.5497202439024391
Crew 5, from 2053-to-2821 : 0.1614543902439024
Crew 2, from 4132-to-5105 : 0.5069934146341463
Crew 3, from 4915-to-3074 : 0.3690682926829268
Crew 1, from 2910-to-3120 : 0.01929573170731707
Crew 4, from 2409-to-1807 : 0.1400990243902439
Crew 6, from 6041-to-4246 : 0.5142221951219511
Crew 5, from 2821-to-5691 : 0.4795865853658536
Crew 3, from 3074-to-1477 : 0.2106853658536585
Crew 2, from 5105-to-3104 : 0.5277941463414632
Crew 1, from 3120-to-5251 : 0.4700641463414635
Crew 4, from 1807-to-1568 : 0.348610731707317
Crew 3, from 1477-to-2440 : 0.2589385365853659
Crew 5, from 5691-to-3679 : 0.4548224390243901
Crew 6, from 4246-to-4584 : 0.3001739024390243
Crew 2, from 3104-to-2543 : 0.08070414634146343
Crew 4, from 1568-to-2094 : 0.2941290243902439
Crew 3, from 2440-to-2408 : 0.04485585365853659
Crew 6, from 4584-to-1989 : 0.3289746341463415
Crew 1, from 5251-to-3

In [13]:
#pd.DataFrame({"crews": new_controls}).to_excel(
#    "new_controls_"+ ds_sel +".xlsx",
#    index=False
#)

## create an INP file with the new controls (test)

In [14]:
import os


def write_inp_controls(input_inp, output_inp, crews, new_controls):
    """
    Creates a new EPANET INP file using the control rules generated by
    allocate_crews().

    Parameters
    ----------
    input_inp : str
        Path to the original damaged INP file.

    output_inp : str
        Path where the modified INP file will be saved.

    crews : dict
        Crew allocation dictionary returned by allocate_crews().
        (Currently only stored for compatibility and future extensions.)

    new_controls : list of str
        List of EPANET control lines generated by allocate_crews().

    Returns
    -------
    str
        Path to the generated INP file.
    """

    # -------------------------------------------------------------
    # Basic checks
    # -------------------------------------------------------------
    if not os.path.exists(input_inp):
        raise FileNotFoundError(f"Input file not found:\n{input_inp}")

    if not isinstance(new_controls, list):
        raise TypeError("new_controls must be a list of strings.")

    if len(new_controls) == 0:
        raise ValueError("new_controls is empty.")

    # -------------------------------------------------------------
    # Read original INP
    # -------------------------------------------------------------
    with open(input_inp, "r") as f:
        lines = f.readlines()

    # -------------------------------------------------------------
    # Locate [CONTROLS]
    # -------------------------------------------------------------
    controls_start = None

    for i, line in enumerate(lines):
        if line.strip().upper() == "[CONTROLS]":
            controls_start = i
            break

    # -------------------------------------------------------------
    # If [CONTROLS] does not exist, create it
    # -------------------------------------------------------------
    if controls_start is None:

        if lines[-1].strip() != "":
            lines.append("\n")

        lines.append("[CONTROLS]\n")
        controls_start = len(lines) - 1

        controls_end = len(lines)

    else:

        # Find the next section
        controls_end = len(lines)

        for j in range(controls_start + 1, len(lines)):

            txt = lines[j].strip()

            if txt.startswith("[") and txt.endswith("]"):
                controls_end = j
                break

    # -------------------------------------------------------------
    # Build new CONTROLS section
    # -------------------------------------------------------------
    control_block = [
        "[CONTROLS]\n",
        "; ---------------------------------------------------------\n",
        "; Restoration controls generated automatically\n",
        "; ---------------------------------------------------------\n",
    ]

    for control in new_controls:

        control = control.rstrip()

        if control != "":
            control_block.append(control + "\n")

    control_block.append("\n")

    # -------------------------------------------------------------
    # Replace old CONTROLS section
    # -------------------------------------------------------------
    new_lines = (
        lines[:controls_start]
        + control_block
        + lines[controls_end:]
    )

    # -------------------------------------------------------------
    # Save new INP
    # -------------------------------------------------------------
    with open(output_inp, "w") as f:
        f.writelines(new_lines)

    print("=" * 60)
    print("Restoration INP successfully created.")
    print(f"Output file : {output_inp}")
    print(f"Controls    : {len(new_controls)}")
    print("=" * 60)

    return output_inp

In [ ]:
# load the wdn as INP file WITH the broken pipes
input_inp = 'BBM-EPS_'+ds_sel+'mcg.inp'

# Export the new INP file with the controls
output_inp="BBM-EPS_"+ds_sel+"_restoration.inp"

In [16]:
controls_inp = write_inp_controls(
    input_inp=input_inp,
    output_inp=output_inp,
    crews=crews,
    new_controls=new_controls
)

print(f"Created: {controls_inp}")

Restoration INP successfully created.
Output file : BBM-EPS_DS2_restoration_6c.inp
Controls    : 424
Created: BBM-EPS_DS2_restoration_6c.inp
